In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup as bs
from selenium.webdriver.common.keys import Keys
import time
import requests as req
from bs4 import BeautifulSoup as bs
import pandas as pd
import numpy as np

## 로그인 및 사이트 입력

In [10]:
op = Options()
op.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36")
driver = webdriver.Chrome(options=op)
driver.get("https://nid.naver.com/nidlogin.login")

user_id = input("아이디:")
user_pw = input("비밀번호:")

driver.execute_script("document.getElementsByName('id')[0].value = arguments[0]", user_id)
driver.execute_script("document.getElementsByName('pw')[0].value = arguments[0]", user_pw)
# 보통 로그인 버튼의 ID는 'log.login'입니다.
driver.find_element(By.ID, "log.login").click()
time.sleep(2)

아이디: 123
비밀번호: 123


In [3]:
url = input("카페 url 입력:")
driver.get(url)
time.sleep(0.5)

카페 url 입력: https://cafe.naver.com/dlxogns01


## 검색어 및 제외할 검색어

In [4]:
search = driver.find_element(By.XPATH, '//*[@id="topLayerQueryInput"]')
search.click()
keyword = input("검색어:")
time.sleep(0.1)
search.send_keys(keyword)
search.send_keys(Keys.ENTER)
time.sleep(0.1)
driver.find_element(By.XPATH, '//*[@id="cafe_content"]/div[1]/div/button').click()
time.sleep(0.1)
except_word = driver.find_element(By.XPATH, '//*[@id="cafe_content"]/div[1]/div/div[5]/div[2]/input')
except_word.click()
except_word.send_keys(input("제외할 단어:"))
driver.find_element(By.XPATH, '//*[@id="cafe_content"]/div[1]/div/div[4]/button').click()

검색어: 고민
제외할 단어: 주식


## 아래의 코드는 반드시 변경해야 합니다!!!

1. 현재 페이지 url 가져오시고 &page=1 부분에서 1을 {i} 로 변경해주세요

driver.get(f'') 이 부분에 url 넣으시면 됩니다

2. 마지막 페이지는 url에서 50 넣어보고 40 넣어보고... 이런 식으로 페이지 찾아주세요

In [5]:
i = 1
url_list = []
while True :
    driver.get(f'https://cafe.naver.com/f-e/cafes/23676262/menus/0?viewType=L&ta=ARTICLE_COMMENT&page=1&q=%EA%B3%A0%EB%AF%BC&eq=%EC%A3%BC%EC%8B%9D')#여기 입력하세요
    time.sleep(1.6)
    soup = bs(driver.page_source, 'html.parser')
    attribute = soup.select('a.article')
    for a in attribute :
        url_list.append(a['href'])
    if i == 3 :#여기 입력하세요
        break
    i += 1

In [6]:
from tqdm import tqdm

In [8]:
# 필수 모듈 임포트
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from selenium.common.exceptions import UnexpectedAlertPresentException, NoAlertPresentException
from bs4 import BeautifulSoup as bs
from tqdm import tqdm

# 데이터 담을 리스트 초기화 (이 부분이 없어서 NameError 발생했음)
title = []
content = []
date = []
view = []
comment = []
failed_urls = []

for i in tqdm(url_list):
    try:
        driver.get(i)
        
        # 1. Alert 발생 여부 확인
        try:
            alert = driver.switch_to.alert
            alert.accept()
            
            # Alert이 떴다는 건 실패했다는 의미이므로 None 처리 후 다음으로 넘어감
            failed_urls.append(i)
            title.append(None)
            date.append(None)
            view.append(None)
            content.append(None)
            comment.append([])
            continue
        except NoAlertPresentException:
            pass

        # 2. 프레임 전환 및 데이터 크롤링
        WebDriverWait(driver, 10).until(
            EC.frame_to_be_available_and_switch_to_it('cafe_main')
        )
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'div > div > h3'))
        )
        
        soup = bs(driver.page_source, 'html.parser')

        # 데이터 추출
        _title   = soup.select('div > div > h3')[0].text
        _date    = soup.select('div.article_info > span.date')[0].text.split()[0]
        _view    = soup.select('div > div.WriterInfo > div.profile_area > div.article_info > span.count')[0].text.split()[-1]
        _content = soup.select('div > div > div.se-module')[0].text
        _comment = [c.get_text(separator='\n').strip() for c in soup.select('div.comment_text_box > p > span.text_comment')]

        # 정상 데이터 추가
        title.append(_title)
        date.append(_date)
        view.append(_view)
        content.append(_content)
        comment.append(_comment)

    except (UnexpectedAlertPresentException, Exception) as e:
        # 3. 예외 발생 시 처리
        if isinstance(e, UnexpectedAlertPresentException):
            try: driver.switch_to.alert.accept()
            except: pass
        
        failed_urls.append(i)
        title.append(None)
        date.append(None)
        view.append(None)
        content.append(None)
        comment.append([])

    finally:
        # 4. 프레임 탈출 (중요)
        # cafe_main 프레임 안에 머물면 다음 루프에서 요소를 못 찾을 수 있음
        try:
            driver.switch_to.default_content()
        except:
            pass

print(f"\n✅ 완료: 성공 {len(url_list) - len(failed_urls)}개 / 실패 {len(failed_urls)}개")
print(f"길이 확인: title={len(title)}, content={len(content)}, date={len(date)}, view={len(view)}, comment={len(comment)}")

100%|██████████████████████████████████████████████████████████████████████████████████| 45/45 [01:21<00:00,  1.81s/it]


In [9]:
df = pd.DataFrame({
    'title': title,
    'content': content,
    'comment' : comment,
    'date': date,
    'view': view,
    'url' : url_list
})
df.head()

,title,content,comment,date,view,url
0,소양천에서 전주천을 돌아 전주를 한바퀴 라이딩해봅니다.,"\n아침에는 눈비가 오더니,낮에는 바람이 제법 불어옵니다.​다음주에 북한강을 거쳐 ...",[북한강을 거처 동해안 라이딩 멋지십니다. ^^\n제가 그동안 소식을 모르는데 학교...,2026.03.09.,138,https://cafe.naver.com/f-e/cafes/23676262/arti...
1,이런 중장년 영어 모임 구상해봅니다^^,"\n​지난 금요일 저녁 7시, 서울에서 중장년 영어 모임을 잘 마쳤습니다.앞으로 이...",[],2026.03.09.,161,https://cafe.naver.com/f-e/cafes/23676262/arti...
2,이 가격의 삼전 닉스는 좋은 가격이라 생각 합니다.,\n\n\n\n,[],2026.03.09.,"1,214",https://cafe.naver.com/f-e/cafes/23676262/arti...
3,죽음을 생각하면,"\n50대 초반입니다.죽기 살기로 달려왔던 삶이였고, 힘들줄도 모르고 생활했던것 같...","[대부분은 남은 인생을 어떻게 하면 더 재미나게 살까 고민하는 것 같은데, 님은 다...",2026.03.09.,438,https://cafe.naver.com/f-e/cafes/23676262/arti...
4,"도전,,,ㅋㅋ",\n 정기건강검진 받던 집 근처 병원에서매번 위 내시경등을 받았으나 2024년 1...,"[동시에 두그릇...부자시네요 ㅎ 맛점 하십시요, 평소 식탐은 없으며 하루 2식이며...",2026.03.09.,339,https://cafe.naver.com/f-e/cafes/23676262/arti...


## 데이터 저장 : 파일명 입력하세요!

In [11]:
df.to_csv('.csv', index=False, encoding='utf-8-sig')
import pickle
with open('.pickle', 'wb') as f:
    pickle.dump(df, f)